# Notebook 2: DeepFM (Deep Factorization Machine)
### Predicting Risky Drug Combinations Using DeepFM in PyTorch

This notebook demonstrates building, training, and benchmarking a **DeepFM** model that combines an FM branch and a deep MLP branch over shared drug embeddings.

---

## 1. Problem
Predict whether combining two drugs produces a **Severe** adverse drug-drug interaction (`Label = 1`) or **Not severe** (`Label = 0`).
DeepFM integrates low-order linear and pairwise interactions (FM component) with high-order non-linear feature interactions (Deep MLP component) using end-to-end learning without manual feature engineering.

In [1]:
import os
import sys
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} | SEED: {SEED}")

/home/natnale/.config/matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-_1_fxlo3 because there was an issue with the default path (/home/natnale/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Using device: cpu | SEED: 42


## 2. Dataset
We use the exact same 40,000 interactions across 420 drugs. The Severe class represents ~16.2% of samples.

In [2]:
data_path = "../data/db_drug_interactions_severity.csv" if os.path.exists("../data/db_drug_interactions_severity.csv") else "db_drug_interactions_severity.csv"
raw_df = pd.read_csv(data_path)
print(f"Total dataset shape: {raw_df.shape}")
print(f"Severe interactions: {(raw_df['Label'] == 1).sum()} ({100 * (raw_df['Label'] == 1).mean():.2f}%)")

Total dataset shape: (40000, 5)
Severe interactions: 6489 (16.22%)


## 3. Leakage Prevention
Input features are strictly `Drug 1` and `Drug 2`. `Interaction Description` and `Severity` are excluded from model inputs. Pair orderings $(A, B)$ and $(B, A)$ are grouped together to prevent cross-split pair leakage.

## 4. Split
To ensure a fair comparison with the NumPy FM model, we load the **exact same split files** generated in Step 1.

In [3]:
train_path = "../data/split_train.csv" if os.path.exists("../data/split_train.csv") else "split_train.csv"
val_path   = "../data/split_val.csv" if os.path.exists("../data/split_val.csv") else "split_val.csv"
test_path  = "../data/split_test.csv" if os.path.exists("../data/split_test.csv") else "split_test.csv"
map_path   = "../data/drug_mapping.json" if os.path.exists("../data/drug_mapping.json") else "drug_mapping.json"

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

with open(map_path) as f:
    drug_to_id = json.load(f)["drug_to_id"]
n_drugs = len(drug_to_id)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Vocabulary: {n_drugs} drugs")

Train: 28002 | Val: 5999 | Test: 5999
Vocabulary: 420 drugs


## 5. Model Mathematics
DeepFM combines the output of an FM branch and a Deep MLP branch over shared embeddings:
$$\hat{y} = z_{\text{FM}} + z_{\text{Deep}}$$

1. **FM Branch**:
   $$z_{\text{FM}} = b + w_1[i] + w_2[j] + \langle v_1, v_2 \rangle$$
2. **Deep Branch**:
   $$a^{(0)} = [v_1; v_2] \in \mathbb{R}^{2k}$$
   $$a^{(l+1)} = \text{ReLU}(W^{(l)} a^{(l)} + b^{(l)})$$
   $$z_{\text{Deep}} = W^{(\text{out})} a^{(\text{last})} + b^{(\text{out})}$$
3. **Probability & Loss**:
   $$p = \sigma(\hat{y})$$
   Trained using `nn.BCEWithLogitsLoss()`.

## 6. Implementation
DeepFM implemented in PyTorch with configurable embedding dimension $k$, hidden layers, and dropout.

In [4]:
class DeepFM(nn.Module):
    def __init__(self, num_drugs, embedding_dim=32, hidden_dims=(64, 32), dropout=0.0, use_fm=True, use_deep=True):
        super().__init__()
        self.num_drugs = num_drugs
        self.embedding_dim = embedding_dim
        self.hidden_dims = tuple(hidden_dims)
        self.dropout_rate = dropout
        self.use_fm = use_fm
        self.use_deep = use_deep

        # Shared Embeddings
        self.embedding = nn.Embedding(num_drugs, embedding_dim)
        self.drug1_embedding = self.embedding
        self.drug2_embedding = self.embedding
        scale = 1.0 / np.sqrt(embedding_dim)
        nn.init.uniform_(self.embedding.weight, -scale, scale)

        # FM Branch
        self.bias = nn.Parameter(torch.zeros(1))
        self.w1 = nn.Embedding(num_drugs, 1)
        self.w2 = nn.Embedding(num_drugs, 1)
        nn.init.zeros_(self.w1.weight)
        nn.init.zeros_(self.w2.weight)

        # Deep Branch
        mlp_layers = []
        in_dim = 2 * embedding_dim
        for h in self.hidden_dims:
            mlp_layers.append(nn.Linear(in_dim, h))
            mlp_layers.append(nn.ReLU())
            if dropout > 0.0:
                mlp_layers.append(nn.Dropout(p=dropout))
            in_dim = h
        mlp_layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*mlp_layers)

    def forward_fm(self, idx1, idx2, v1, v2):
        w1_val = self.w1(idx1).squeeze(-1)
        w2_val = self.w2(idx2).squeeze(-1)
        dot_val = torch.sum(v1 * v2, dim=-1)
        return self.bias + w1_val + w2_val + dot_val

    def forward_deep(self, v1, v2):
        concat_emb = torch.cat([v1, v2], dim=-1)
        return self.mlp(concat_emb).squeeze(-1)

    def forward(self, idx1, idx2):
        v1, v2 = self.embedding(idx1), self.embedding(idx2)
        if self.use_fm and self.use_deep:
            return self.forward_fm(idx1, idx2, v1, v2) + self.forward_deep(v1, v2)
        elif self.use_fm:
            return self.forward_fm(idx1, idx2, v1, v2)
        elif self.use_deep:
            return self.forward_deep(v1, v2)
        raise ValueError("At least one branch must be active.")

    def forward_components(self, idx1, idx2):
        v1, v2 = self.embedding(idx1), self.embedding(idx2)
        fm_l = self.forward_fm(idx1, idx2, v1, v2) if self.use_fm else torch.zeros_like(idx1, dtype=torch.float32)
        dp_l = self.forward_deep(v1, v2) if self.use_deep else torch.zeros_like(idx1, dtype=torch.float32)
        final_l = fm_l + dp_l
        return {"fm_logit": fm_l, "deep_logit": dp_l, "final_logit": final_l, "probability": torch.sigmoid(final_l)}

print("DeepFM PyTorch module defined.")

DeepFM PyTorch module defined.


## 7. Training
DataLoaders are built with batch size = 256. The model trains with Adam (lr=0.001) and `BCEWithLogitsLoss`.

In [5]:
class DrugInteractionDataset(Dataset):
    def __init__(self, df, mapping):
        self.i1 = torch.tensor(df["Drug 1"].map(mapping).values, dtype=torch.long)
        self.i2 = torch.tensor(df["Drug 2"].map(mapping).values, dtype=torch.long)
        self.y  = torch.tensor(df["Label"].values, dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.i1[idx], self.i2[idx], self.y[idx]

train_loader = DataLoader(DrugInteractionDataset(train_df, drug_to_id), batch_size=256, shuffle=True)
val_loader   = DataLoader(DrugInteractionDataset(val_df, drug_to_id), batch_size=256, shuffle=False)
test_loader  = DataLoader(DrugInteractionDataset(test_df, drug_to_id), batch_size=256, shuffle=False)

def eval_loader(model, loader, threshold=0.5):
    model.eval()
    crit = nn.BCEWithLogitsLoss()
    tot_loss, tot_n = 0.0, 0
    all_probs, all_y = [], []
    with torch.no_grad():
        for i1, i2, y in loader:
            i1, i2, y = i1.to(device), i2.to(device), y.to(device)
            out = model(i1, i2)
            tot_loss += crit(out, y).item() * len(y)
            tot_n += len(y)
            all_probs.append(torch.sigmoid(out).cpu().numpy())
            all_y.append(y.cpu().numpy())
    probs = np.concatenate(all_probs)
    y_true = np.concatenate(all_y)
    preds = (probs >= threshold).astype(float)
    return {
        "loss": tot_loss / max(tot_n, 1),
        "acc": accuracy_score(y_true, preds),
        "prec": precision_score(y_true, preds, zero_division=0),
        "rec": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "cm": confusion_matrix(y_true, preds),
        "probs": probs,
        "y_true": y_true
    }

print("Training routines ready.")

Training routines ready.


## 8. Benchmarks
Controlled sequential benchmark:
1. **Embedding dimension**: $k \in [4, 8, 16, 32]$
2. **MLP architecture**: `[32]`, `[64]`, `[64, 32]`
3. **Dropout rate**: `0.0`, `0.2`, `0.5`

In [6]:
# Load precomputed benchmark results from results/deepfm_experiments.csv
csv_path = "../results/deepfm_experiments.csv" if os.path.exists("../results/deepfm_experiments.csv") else "deepfm_experiments.csv"
bench_df = pd.read_csv(csv_path)
display(bench_df[["stage", "k", "hidden_layers", "dropout", "val_loss", "val_accuracy", "val_f1", "n_params"]])

,stage,k,hidden_layers,dropout,val_loss,val_accuracy,val_f1,n_params
0,1_embedding_dim,4,"(32,)",0.0,0.152206,0.942157,0.824304,2842
1,1_embedding_dim,8,"(32,)",0.0,0.142935,0.946491,0.838124,4778
2,1_embedding_dim,16,"(32,)",0.0,0.124561,0.946324,0.844143,8650
3,1_embedding_dim,32,"(32,)",0.0,0.122222,0.951659,0.849272,16394
4,2_mlp_architecture,32,"(32,)",0.0,0.122222,0.951659,0.849272,16394
5,2_mlp_architecture,32,"(64,)",0.0,0.117294,0.954659,0.859649,18506
6,2_mlp_architecture,32,"(64, 32)",0.0,0.099008,0.966828,0.897158,20554
7,3_dropout,32,"(64, 32)",0.0,0.099008,0.966828,0.897158,20554
8,3_dropout,32,"(64, 32)",0.2,0.098734,0.965328,0.893986,20554
9,3_dropout,32,"(64, 32)",0.5,0.115959,0.956993,0.869168,20554


## 9. Model Selection & Ablation
The optimal configuration is **$k = 32$**, **MLP = `[64, 32]`**, **Dropout = 0.0** (Val F1: **0.8972**).
We compare FM-only, Deep-only, and Full DeepFM to quantify the contribution of each branch.

In [7]:
abl_path = "../results/ablation.csv" if os.path.exists("../results/ablation.csv") else "ablation.csv"
abl_df = pd.read_csv(abl_path)
display(abl_df[["variant", "val_loss", "accuracy", "precision", "recall", "f1"]])

,variant,val_loss,accuracy,precision,recall,f1
0,FM-only,0.151687,0.955326,0.844575,0.887975,0.865731
1,Deep-only,0.112668,0.957993,0.855172,0.892086,0.873239
2,Full DeepFM,0.100747,0.966161,0.905263,0.883864,0.894436


## 10. Final Test Results & FM Comparison
We evaluate the selected DeepFM model **once** on the test set and compare it directly against the FM model from Notebook 1.

In [8]:
# Load best checkpoint
best_ckpt = "../models/deepfm/deepfm_best.pt" if os.path.exists("../models/deepfm/deepfm_best.pt") else "deepfm_best.pt"
checkpoint = torch.load(best_ckpt, map_location=device, weights_only=False)

best_deepfm = DeepFM(num_drugs=n_drugs, embedding_dim=32, hidden_dims=(64, 32), dropout=0.0)
best_deepfm.load_state_dict(checkpoint["model_state_dict"], strict=False)
best_deepfm.eval()

test_threshold = 0.50
test_m = eval_loader(best_deepfm, test_loader, threshold=test_threshold)
tn, fp, fn, tp = test_m["cm"].ravel()

print("=" * 50)
print("FINAL DEEPFM TEST RESULTS")
print("=" * 50)
print(f"  Accuracy  : {test_m['acc']:.4f}")
print(f"  Precision : {test_m['prec']:.4f}")
print(f"  Recall    : {test_m['rec']:.4f}")
print(f"  F1 Score  : {test_m['f1']:.4f}")
print(f"  Loss      : {test_m['loss']:.4f}")
print(f"  TP={tp}, TN={tn}, FP={fp}, FN={fn}")

# Comparison table
cmp_path = "../results/final_comparison.csv" if os.path.exists("../results/final_comparison.csv") else "final_comparison.csv"
if os.path.exists(cmp_path):
    print("\n" + "=" * 50)
    print("FM vs DeepFM Final Test Comparison")
    print("=" * 50)
    display(pd.read_csv(cmp_path)[["model", "test_accuracy", "test_precision", "test_recall", "test_f1", "tp", "fp"]])

FINAL DEEPFM TEST RESULTS
  Accuracy  : 0.9697
  Precision : 0.9149
  Recall    : 0.8961
  F1 Score  : 0.9054
  Loss      : 0.0930
  TP=871, TN=4946, FP=81, FN=101

FM vs DeepFM Final Test Comparison


,model,test_accuracy,test_precision,test_recall,test_f1,tp,fp
0,FM (from scratch),0.373062,0.195324,0.919753,0.322220,894,3683
1,DeepFM,0.969662,0.914916,0.896091,0.905405,871,81


## 11. Explanation of One Drug Pair
Evaluating the same pair examined in Notebook 1 (**Dronedarone + Atenolol**) to compare FM and Deep contributions.

In [9]:
d1, d2 = "Dronedarone", "Atenolol"
t_i1 = torch.tensor([drug_to_id[d1]], dtype=torch.long, device=device)
t_i2 = torch.tensor([drug_to_id[d2]], dtype=torch.long, device=device)

with torch.no_grad():
    comps = best_deepfm.forward_components(t_i1, t_i2)
    fm_part   = float(comps["fm_logit"].item())
    deep_part = float(comps["deep_logit"].item())
    final_log = float(comps["final_logit"].item())
    final_p   = float(comps["probability"].item())

print("=" * 50)
print(f"DeepFM Prediction Breakdown: {d1} + {d2}")
print("=" * 50)
print(f"  FM branch contribution   : {fm_part:+.4f}")
print(f"  Deep branch contribution : {deep_part:+.4f}")
print(f"  Combined Logit           : {final_log:+.4f}")
print(f"  Probability P(Severe)    : {final_p:.4f}")
print(f"  Prediction (>= 0.50)     : {'Severe (1)' if final_p >= 0.50 else 'Not Severe (0)'}")
print(f"  Agreement with FM        : DeepFM agrees with FM, predicting high risk for Antiarrhythmic + Beta-blocker.")

DeepFM Prediction Breakdown: Dronedarone + Atenolol
  FM branch contribution   : +0.0272
  Deep branch contribution : +6.0289
  Combined Logit           : +6.0560
  Probability P(Severe)    : 0.9977
  Prediction (>= 0.50)     : Severe (1)
  Agreement with FM        : DeepFM agrees with FM, predicting high risk for Antiarrhythmic + Beta-blocker.


## 12. Live Demonstration
Loads the frozen `deepfm_best.pt` checkpoint and performs inference on any drug pair **without retraining**.

In [10]:
# Load checkpoint from disk without retraining
demo_ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
demo_model = DeepFM(num_drugs=n_drugs, embedding_dim=32, hidden_dims=(64, 32), dropout=0.0)
demo_model.load_state_dict(demo_ckpt["model_state_dict"], strict=False)
demo_model.eval()

def live_deepfm_predict(drug1, drug2, threshold=0.50):
    if drug1 not in drug_to_id or drug2 not in drug_to_id:
        return f"One or both drugs not found."
    idx1 = torch.tensor([drug_to_id[drug1]], dtype=torch.long, device=device)
    idx2 = torch.tensor([drug_to_id[drug2]], dtype=torch.long, device=device)
    with torch.no_grad():
        comps = demo_model.forward_components(idx1, idx2)
        p = float(comps["probability"].item())
        fm_val = float(comps["fm_logit"].item())
        deep_val = float(comps["deep_logit"].item())
    risk = "Severe (High Risk)" if p >= threshold else "Not Severe (Low/Moderate Risk)"
    return {
        "Drug 1": drug1,
        "Drug 2": drug2,
        "FM Logit": round(fm_val, 3),
        "Deep Logit": round(deep_val, 3),
        "P(Severe)": round(p, 4),
        "Prediction": risk
    }

demo_pairs = [
    ("Dronedarone", "Atenolol"),
    ("Methadone", "Sulfisoxazole"),
    ("Abiraterone", "Acebutolol"),
    ("Paliperidone", "Chloroquine"),
]

print(f"Loaded frozen DeepFM model from {best_ckpt}. Live predictions:")
pd.DataFrame([live_deepfm_predict(d1, d2) for d1, d2 in demo_pairs])

Loaded frozen DeepFM model from ../models/deepfm/deepfm_best.pt. Live predictions:


,Drug 1,Drug 2,FM Logit,Deep Logit,P(Severe),Prediction
0,Dronedarone,Atenolol,0.027,6.029,0.9977,Severe (High Risk)
1,Methadone,Sulfisoxazole,0.337,1.845,0.8986,Severe (High Risk)
2,Abiraterone,Acebutolol,-0.312,-11.219,0.0000,Not Severe (Low/Moderate Risk)
3,Paliperidone,Chloroquine,0.206,3.291,0.9706,Severe (High Risk)
